# Convert Parquet to NDJSON with chDB

Companion to [How to convert Parquet to NDJSON](https://clickhouse.com/resources/engineering/convert-parquet-to-ndjson).

Run `./generate.sh` first to create `data/events.parquet`. chDB runs the same ClickHouse engine in-process, so the SQL is identical to the `clickhouse local` one-liner.

In [ ]:
import os
import chdb

os.chdir("data")

## Convert Parquet -> NDJSON

Same `SELECT ... INTO OUTFILE ... FORMAT JSONEachRow` as the CLI. The schema is read from the Parquet footer; types carry over.

In [ ]:
chdb.query(
    "SELECT * FROM file('events.parquet') "
    "INTO OUTFILE 'events_py.ndjson' TRUNCATE FORMAT JSONEachRow"
)

with open("events_py.ndjson") as f:
    for _, line in zip(range(3), f):
        print(line.rstrip())

## Or capture NDJSON as a string

Drop `INTO OUTFILE` and chDB returns the formatted bytes you can stream anywhere.

In [ ]:
ndjson = chdb.query(
    "SELECT event_id, country, amount FROM file('events.parquet') "
    "WHERE action = 'purchase' ORDER BY amount DESC LIMIT 2 FORMAT JSONEachRow"
)
print(str(ndjson).rstrip())